# Batch Evaluation Confusion Table

This notebook loads JSON result files from the `further_evaluation/batch_evaluation` folder, aggregates results for a selected batch size, and shows a 3x2 table: rows are model *predictions* (`New table`, `Correct existing table`, `Wrong existing table`) and columns are the *actual* ground-truth (`New table`, `Existing table`).

Correct prediction cells are highlighted green; other cells are red. Use the dropdown to select a batch size.

In [13]:
import os
import re
import json
import glob
import pandas as pd
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

from util.insert_parser import parse_insert, UnexpectedTokenException

BASE_DIR = os.path.join(os.getcwd(), 'further_evaluation', 'batch_evaluation')

ROW_NAMES = ['New table', 'Correct existing table', 'Wrong existing table']
PRIMARY_COLS = ['New table', 'Existing table']
SUBCOLS = ['Exact name', 'Synonymous name', 'Undefined']

def find_batch_sizes():
    pattern = re.compile(r'_batch_?(\d+).json$', re.IGNORECASE)
    sizes = set()
    if not os.path.isdir(BASE_DIR):
        return []
    for root, dirs, files in os.walk(BASE_DIR):
        for fn in files:
            m = pattern.search(fn)
            if m:
                sizes.add(int(m.group(1)))
    return sorted(sizes)

def load_counts_for_batch(batch_size, column_filter='All'):
    # initialize counts DataFrame with MultiIndex columns (primary, subcol)
    tuples = []
    for p in PRIMARY_COLS:
        for s in SUBCOLS:
            tuples.append((p, s))
    col_index = pd.MultiIndex.from_tuples(tuples)
    counts = pd.DataFrame(0, index=ROW_NAMES, columns=col_index)

    # glob all files that contain _batch_{size}.json under BASE_DIR (recursively)
    pattern = os.path.join(BASE_DIR, '**', f'*_batch_{batch_size}.json')
    files = glob.glob(pattern, recursive=True)
    if not files:
        return counts

    for fp in files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception:
            # skip unreadable files
            continue
        if not isinstance(data, list):
            continue
        for entry in data:
            expected = entry.get('expected_table_name')
            predicted = entry.get('predicted_table_name')
            expected_columns = entry.get('expected_column_names') or []
            db_state = entry.get('database_state') or {}
            insert_sql = entry.get('insert', '') or ''
            # determine actual (is expected in DB state?)
            actual_is_existing = False
            try:
                if isinstance(db_state, dict):
                    actual_is_existing = (expected in db_state)
            except Exception:
                actual_is_existing = False

            # determine predicted category (rows)
            predicted_is_existing = False
            try:
                if isinstance(db_state, dict):
                    predicted_is_existing = (predicted in db_state)
            except Exception:
                predicted_is_existing = False

            if not predicted_is_existing:
                row = 'New table'
            else:
                if predicted == expected:
                    row = 'Correct existing table'
                else:
                    row = 'Wrong existing table'

            col_primary = 'Existing table' if actual_is_existing else 'New table'

            # parse insert to extract table name and columns if present
            insert_table = None
            insert_columns = None
            try:
                if isinstance(insert_sql, str) and insert_sql.strip() != '':
                    parsed = parse_insert(insert_sql.lower())
                    insert_table = parsed.get('table')
                    insert_columns = parsed.get('columns')
            except UnexpectedTokenException:
                insert_table = None
                insert_columns = None
            except Exception:
                insert_table = None
                insert_columns = None

            # determine subcategory for table-name (existing behavior)
            if not insert_table:
                table_sub = 'Undefined'
            else:
                try:
                    if expected is not None and str(insert_table).lower() == str(expected).lower():
                        table_sub = 'Exact name'
                    else:
                        table_sub = 'Synonymous name'
                except Exception:
                    table_sub = 'Synonymous name'

            # determine subcategory for columns per entry (use first column if present)
            if not insert_columns or len(insert_columns) == 0:
                colname_sub = 'Undefined'
            else:
                first_insert_col = str(insert_columns[0])
                if expected_columns and len(expected_columns) > 0:
                    try:
                        if first_insert_col.lower() == str(expected_columns[0]).lower():
                            colname_sub = 'Exact name'
                        else:
                            colname_sub = 'Synonymous name'
                    except Exception:
                        colname_sub = 'Synonymous name'
                else:
                    # no expected columns to compare to, treat as Synonymous if present
                    colname_sub = 'Synonymous name'

            # apply column filter: if not 'All' and column subcategory doesn't match, skip entry
            if column_filter and column_filter != 'All' and colname_sub != column_filter:
                continue

            # increment counts for the table-name subcategory (the subcolumn axis)
            sub = table_sub
            if (col_primary, sub) in counts.columns and row in counts.index:
                counts.loc[row, (col_primary, sub)] += 1
    return counts

def compute_plot_data(batch_size, col_tuple, column_filter='All'):
    # col_tuple is (primary, sub) like ('Existing table','Exact name')
    # returns mapping from db_size -> (total_in_col, correct_in_col)
    totals = {}
    corrects = {}
    pattern = os.path.join(BASE_DIR, '**', f'*_batch_{batch_size}.json')
    files = glob.glob(pattern, recursive=True)
    if not files:
        return pd.DataFrame(columns=['db_count','percent_correct']).set_index('db_count')

    for fp in files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception:
            continue
        if not isinstance(data, list):
            continue
        for entry in data:
            expected = entry.get('expected_table_name')
            predicted = entry.get('predicted_table_name')
            expected_columns = entry.get('expected_column_names') or []
            db_state = entry.get('database_state') or {}
            insert_sql = entry.get('insert', '') or ''
            # db size = number of tables in database_state (dict keys)
            db_size = 0
            try:
                if isinstance(db_state, dict):
                    db_size = len(db_state)
            except Exception:
                db_size = 0

            # determine predicted category (rows)
            predicted_is_existing = False
            try:
                if isinstance(db_state, dict):
                    predicted_is_existing = (predicted in db_state)
            except Exception:
                predicted_is_existing = False

            if not predicted_is_existing:
                row = 'New table'
            else:
                if predicted == expected:
                    row = 'Correct existing table'
                else:
                    row = 'Wrong existing table'

            col_primary = 'Existing table' if (isinstance(db_state, dict) and expected in db_state) else 'New table'

            # parse insert for table and columns
            insert_table = None
            insert_columns = None
            try:
                if isinstance(insert_sql, str) and insert_sql.strip() != '':
                    parsed = parse_insert(insert_sql.lower())
                    insert_table = parsed.get('table')
                    insert_columns = parsed.get('columns')
            except Exception:
                insert_table = None
                insert_columns = None

            # determine table_sub like in load_counts_for_batch
            if not insert_table:
                table_sub = 'Undefined'
            else:
                try:
                    if expected is not None and str(insert_table).lower() == str(expected).lower():
                        table_sub = 'Exact name'
                    else:
                        table_sub = 'Synonymous name'
                except Exception:
                    table_sub = 'Synonymous name'

            # determine column subcategory
            if not insert_columns or len(insert_columns) == 0:
                colname_sub = 'Undefined'
            else:
                first_insert_col = str(insert_columns[0])
                if expected_columns and len(expected_columns) > 0:
                    try:
                        if first_insert_col.lower() == str(expected_columns[0]).lower():
                            colname_sub = 'Exact name'
                        else:
                            colname_sub = 'Synonymous name'
                    except Exception:
                        colname_sub = 'Synonymous name'
                else:
                    colname_sub = 'Synonymous name'

            # apply column filter (for columns) - skip if not match
            if column_filter and column_filter != 'All' and colname_sub != column_filter:
                continue

            # check if this entry falls into the selected col_tuple
            if (col_primary, table_sub) != tuple(col_tuple):
                # only count entries that belong to the selected confusion column
                continue

            # increment totals and corrects for this db_size
            totals[db_size] = totals.get(db_size, 0) + 1
            # correct if (row=='New table' and primary=='New table') or (row=='Correct existing table' and primary=='Existing table')
            is_correct = (row == 'New table' and col_primary == 'New table') or (row == 'Correct existing table' and col_primary == 'Existing table')
            if is_correct:
                corrects[db_size] = corrects.get(db_size, 0) + 1
    # build DataFrame with db_count sorted
    db_counts = sorted(set(list(totals.keys()) + list(corrects.keys())))
    rows = []
    for d in db_counts:
        tot = totals.get(d, 0)
        corr = corrects.get(d, 0)
        pct = (corr / tot * 100.0) if tot > 0 else np.nan
        rows.append({'db_count': d, 'percent_correct': pct, 'total': tot, 'correct': corr})
    return pd.DataFrame(rows).set_index('db_count')

def render_confusion_html(df, as_percent=True):
    # df has MultiIndex columns (primary, subcol)
    display_df = df.copy()
    # If percentage mode, convert counts to percentages per column (sum over rows)
    if as_percent:
        pct_df = pd.DataFrame(index=df.index, columns=df.columns, dtype=float)
        for col in df.columns:
            col_vals = df[col].astype(float)
            total = col_vals.sum()
            if total == 0:
                pct = [0.0 for _ in col_vals]
            else:
                raw = (col_vals / total) * 100.0
                rounded = [round(x, 1) for x in raw]
                pct = rounded
            for i, rname in enumerate(df.index):
                pct_df.loc[rname, col] = pct[i]
        display_df = pct_df

    # Build HTML table
    total_cols = len(display_df.columns)
    html = []
    html.append('<table style="border-collapse: collapse; font-family: Arial;">')
    html.append('<thead>')
    # top header row: overall label for correct values
    html.append('<tr>')
    html.append('<th style="border: 1px solid #ddd; padding: 6px; background: white;"></th>')
    html.append(f'<th colspan="{total_cols}" style="border: 1px solid #ddd; padding: 6px; background: #f0f0f0; text-align: center;">Correct values</th>')
    html.append('</tr>')
    # second header row: primary groups
    html.append('<tr>')
    html.append('<th style="border: 1px solid #ddd; padding: 6px; background: #f7f7f7; text-align: center;">Predictions</th>')
    primaries = list(dict.fromkeys([c[0] for c in display_df.columns]))
    for p in primaries:
        span = sum(1 for c in display_df.columns if c[0] == p)
        html.append(f'<th colspan="{span}" style="border: 1px solid #ddd; padding: 6px; background: #f7f7f7; text-align: center;">{p}</th>')
    html.append('</tr>')
    # third header row: subcolumn labels
    html.append('<tr>')
    html.append('<th style="border: 1px solid #ddd; padding: 6px; background: #fff;"></th>')
    for _, sub in display_df.columns:
        html.append(f'<th style="border: 1px solid #ddd; padding: 6px; background: #fff; text-align: center;">{sub}</th>')
    html.append('</tr>')
    html.append('</thead>')
    # body rows
    html.append('<tbody>')
    for r in display_df.index:
        html.append('<tr>')
        html.append(f'<th style="border: 1px solid #ddd; padding: 6px; background: #fafafa; text-align: left;">{r}</th>')
        for col in display_df.columns:
            raw_val = display_df.loc[r, col]
            # format value based on mode
            if as_percent:
                val_str = f"{raw_val:.1f}%"
            else:
                val_str = f"{int(df.loc[r, col])}"
            # mark correctness based on primary (actual) match
            is_impossible = (r == 'Correct existing table' and col[0] == 'New table') or (col[0] == 'New table' and col[1] == 'Synonymous name')
            is_correct = (r == 'New table' and col[0] == 'New table') or (r == 'Correct existing table' and col[0] == 'Existing table')
            bg = 'grey' if is_impossible else 'lightgreen' if is_correct else 'salmon'
            html.append(f'<td style="border: 1px solid #ddd; width: 130px; padding: 8px; text-align: center; background-color: {bg};">{val_str}</td>')
        html.append('</tr>')
    html.append('</tbody>')
    html.append('</table>')
    return '\n'.join(html)

In [ ]:
# discover available batch sizes and create dropdown
batch_sizes = find_batch_sizes()
if not batch_sizes:
    print('No batch files found under', BASE_DIR)

batch_dropdown = widgets.Dropdown(
    options=batch_sizes,
    value=batch_sizes[0] if batch_sizes else None,
    description='Batch:',
    )
# Checkbox: default False -> show percentages by default
abs_checkbox = widgets.Checkbox(value=False, description='Absolute numbers')
# Column filter dropdown: All + subcategories
col_filter_dropdown = widgets.Dropdown(options=['All'] + SUBCOLS, value='All', description='Column filter:')
# Plot column dropdown: populate from primary x sub combinations
plot_options = [f'{p} - {s}' for p in PRIMARY_COLS for s in SUBCOLS if not (p == "New table" and s == "Synonymous name")]
plot_col_dropdown = widgets.Dropdown(options=plot_options, value=plot_options[0], description='Plot column:')
output = widgets.Output()

def update_table():
    with output:
        clear_output(wait=True)
        batch = batch_dropdown.value
        col_filter = col_filter_dropdown.value
        counts = load_counts_for_batch(batch, column_filter=col_filter)
        if counts.values.sum() == 0:
            print(f'No entries for batch {batch} (filter={col_filter})')
            display(counts)
            return
        as_percent = not bool(abs_checkbox.value)
        # render with extra headers labeling Actual and Predictions
        html = render_confusion_html(counts, as_percent=as_percent)
        display(HTML(html))

        # Plotting: compute plot data for all batch sizes and selected column
        sel = plot_col_dropdown.value
        primary, sub = [s.strip() for s in sel.split(' - ', 1)]
        
        # Plot percent_correct vs db_count for all batch sizes
        plt.figure(figsize=(10,5))
        
        # Define distinct colors for each batch size
        batch_colors = [('blue', 'lightblue'), ('green', 'lightgreen'), ('darkmagenta', 'magenta')]
        batch_color_map = {b: batch_colors[i % len(batch_colors)] for i, b in enumerate(batch_sizes)}
        
        # collect data from all batches
        all_empty = True
        for b in batch_sizes:
            df_plot = compute_plot_data(b, (primary, sub), column_filter=col_filter)
            if not df_plot.empty:
                all_empty = False
                x = df_plot.index.values
                y = df_plot['percent_correct'].values
                x, y = zip(*[(pair[0], pair[1]) for pair in zip(x, y, df_plot['total'].values) if pair[2] >= 10])
                batch_color = batch_color_map[b]
                # colors = [batch_color[0] if t >= 10 else batch_color[1] for t in df_plot['total'].values]
                # plot line
                plt.plot(x, y, linestyle='-', color=batch_color[0], alpha=0.5)
                # plt.scatter(x, y, marker='o', color=colors)
        
        if all_empty:
            print('No data for selected plot column')
            plt.close()
            return
        
        # Create custom legend for batch sizes
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=batch_color_map[b][0], label=f'Batch {b}') for b in batch_sizes]
        plt.legend(handles=legend_elements, loc='best')
        
        plt.xlabel('Number of tables in database state')
        plt.ylabel('Percent correct (%)')
        plt.title(f"Percent correct: {primary} - {sub} (table), {col_filter} (columns)")
        plt.grid(True)
        plt.ylim(-5,105)
        display(HTML('<div></div>'))
        display(plt.gcf())
        plt.close()

def on_change(change):
    if change['name'] != 'value':
        return
    update_table()

# observe dropdowns and checkbox
batch_dropdown.observe(on_change)
abs_checkbox.observe(on_change)
col_filter_dropdown.observe(on_change)
plot_col_dropdown.observe(on_change)
display(widgets.HBox([batch_dropdown, abs_checkbox, col_filter_dropdown, plot_col_dropdown]))
display(output)

update_table()

Output()

<Figure size 1000x500 with 0 Axes>

<Figure size 1000x500 with 0 Axes>

<Figure size 1000x500 with 0 Axes>

<Figure size 1000x500 with 0 Axes>